# PII Catalog Helpers

Loads `pipeline_configs/pii/pii_config.json`, applies Unity Catalog column tags
(`pii_direct` / `pii_indirect`) to all Silver tables, and populates
`monitoring.pii_column_registry`.

**Run once after any change to pii_config.json.** Safe to re-run — tags are idempotent
and registry rows are MERGE'd.

## Parameters
- `PII_CONFIG_PATH` — DBFS/Volume path to pii_config.json
- `CATALOG` — Unity Catalog name (default `workspace`)
- `DRY_RUN` — if `true`, print actions without applying tags or writing registry

In [ ]:
# ---------------------------------------------------------------------------
# Parameters (override via Databricks job widgets or %run arguments)
# ---------------------------------------------------------------------------
dbutils.widgets.text("PII_CONFIG_PATH", "/Volumes/workspace/default/mnt/pipeline_configs/pii/pii_config.json")
dbutils.widgets.text("CATALOG", "workspace")
dbutils.widgets.dropdown("DRY_RUN", "false", ["true", "false"])

PII_CONFIG_PATH = dbutils.widgets.get("PII_CONFIG_PATH")
CATALOG         = dbutils.widgets.get("CATALOG")
DRY_RUN         = dbutils.widgets.get("DRY_RUN").lower() == "true"

print(f"PII_CONFIG_PATH : {PII_CONFIG_PATH}")
print(f"CATALOG         : {CATALOG}")
print(f"DRY_RUN         : {DRY_RUN}")

In [ ]:
import json
from datetime import datetime, timezone
from pyspark.sql import Row

# ---------------------------------------------------------------------------
# Load pii_config.json
# ---------------------------------------------------------------------------
with open(PII_CONFIG_PATH, "r") as fh:
    pii_config = json.load(fh)

# Only columns that have an explicit sensitivity (not wildcard '*')
tagged_columns = [e for e in pii_config if e["column"] != "*"]

print(f"Loaded {len(pii_config)} pii_config entries, {len(tagged_columns)} explicit column tags")

In [ ]:
# ---------------------------------------------------------------------------
# Apply Unity Catalog column tags
# ---------------------------------------------------------------------------
# Tag map: sensitivity level → tag key + value pair applied to the column
SENSITIVITY_TAG = {
    "pii_direct":   ("pii", "direct"),
    "pii_indirect": ("pii", "indirect"),
}

tagged_ok   = []
tag_skipped = []
tag_errors  = []

for entry in tagged_columns:
    table      = entry["table"]        # e.g. "silver.silver_customer"
    column     = entry["column"]
    sensitivity = entry["sensitivity"]

    if sensitivity not in SENSITIVITY_TAG:
        tag_skipped.append((table, column, sensitivity))
        continue

    tag_key, tag_value = SENSITIVITY_TAG[sensitivity]
    fqn = f"{CATALOG}.{table}"        # e.g. workspace.silver.silver_customer
    sql = f"ALTER TABLE {fqn} ALTER COLUMN {column} SET TAGS ('{tag_key}' = '{tag_value}')"

    if DRY_RUN:
        print(f"[DRY RUN] {sql}")
        tagged_ok.append(entry)
    else:
        try:
            spark.sql(sql)
            print(f"Tagged  {fqn}.{column}  → {tag_key}={tag_value}")
            tagged_ok.append(entry)
        except Exception as exc:
            print(f"ERROR tagging {fqn}.{column}: {exc}")
            tag_errors.append((fqn, column, str(exc)))

print(f"\nTagging summary — OK: {len(tagged_ok)}, skipped: {len(tag_skipped)}, errors: {len(tag_errors)}")
if tag_errors:
    raise RuntimeError(f"{len(tag_errors)} column tag(s) failed — see output above")

In [ ]:
# ---------------------------------------------------------------------------
# Write to monitoring.pii_column_registry
# ---------------------------------------------------------------------------
now = datetime.now(timezone.utc)

rows = [
    Row(
        table_name     = entry["table"],
        column_name    = entry["column"],
        sensitivity    = entry["sensitivity"],
        subject_id_col = entry["subject_id_col"],
        encrypt        = entry["encrypt"],
        tagged_at      = now,
    )
    for entry in pii_config
]

registry_df = spark.createDataFrame(rows)

if DRY_RUN:
    print("[DRY RUN] Would write the following rows to monitoring.pii_column_registry:")
    registry_df.show(truncate=False)
else:
    from delta.tables import DeltaTable

    registry_fqn = f"{CATALOG}.monitoring.pii_column_registry"

    (
        DeltaTable.forName(spark, registry_fqn)
        .alias("t")
        .merge(
            registry_df.alias("s"),
            "t.table_name = s.table_name AND t.column_name = s.column_name",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"pii_column_registry updated — {len(rows)} entries")

In [ ]:
# ---------------------------------------------------------------------------
# Verification — spot check silver_customer.email tag
# ---------------------------------------------------------------------------
if not DRY_RUN:
    result = spark.sql(
        f"DESCRIBE DETAIL {CATALOG}.silver.silver_customer"
    )
    tag_check = spark.sql(
        f"""
        SELECT column_name, tag_name, tag_value
        FROM {CATALOG}.information_schema.column_tags
        WHERE table_schema = 'silver'
          AND table_name   = 'silver_customer'
          AND column_name  = 'email'
        """
    )
    print("UC tags on silver_customer.email:")
    tag_check.show()
    registry_count = spark.sql(
        f"SELECT COUNT(*) AS cnt FROM {CATALOG}.monitoring.pii_column_registry"
    ).collect()[0]["cnt"]
    print(f"pii_column_registry row count: {registry_count}")